# Part 7: Quantum machine learning

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 7.1 What quantum machine learning is

### Logistic regression, trained by gradient descent

The classical classifier from this chapter, written out in NumPy: p(red) = σ(w·x + b), the cross-entropy loss, its gradient, and the update θ ← θ − η∇L. The data are two clusters, with separate training and test points.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=11)


def make_clusters(n):
    y = np.arange(n) % 2                                   # alternate blue (0) and red (1)
    centres = np.where(y[:, None] == 1, [0.45, -0.35], [-0.40, 0.40])
    return centres + 0.28 * rng.normal(size=(n, 2)), y


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


X_train, y_train = make_clusters(60)
X_test, y_test = make_clusters(60)
w, b, lr = np.zeros(2), 0.0, 0.5

for step in range(201):
    p = sigmoid(X_train @ w + b)                           # p(red) for every training point
    loss = -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))
    if step % 40 == 0:
        train_acc = np.mean((p > 0.5) == y_train)
        test_acc = np.mean((sigmoid(X_test @ w + b) > 0.5) == y_test)
        print(f"step {step:3d}: loss {loss:.3f}   train accuracy {train_acc:.0%}   test accuracy {test_acc:.0%}")
    grad_w = X_train.T @ (p - y_train) / len(y_train)
    grad_b = np.mean(p - y_train)
    w, b = w - lr * grad_w, b - lr * grad_b

print("weights", w.round(2), " bias", round(b, 2))

The loss falls step by step and the accuracy on the held-back test points follows the training accuracy, so the model is not overfitting. The gradient formulas are the standard ones for this model: ∂L/∂w = average of (p − y)x, and ∂L/∂b = average of (p − y).

### Exercise 7.1.1: Gradient descent by hand, then by computer

Minimize L(θ) = (θ − 3)² by gradient descent, starting at θ = 0 with η = 0.1. Print θ after each of the first eight steps and check the chapter's numbers 0.6, 1.08 and 1.464. What fraction of the remaining distance to 3 does each step remove?

<details><summary>Hint</summary>

The gradient is dL/dθ = 2(θ − 3), and one step is `theta = theta - eta * grad`.

</details>

In [ ]:
# Your code here

### Exercise 7.1.2: Features decide what a model can learn

Make the circle dataset: points spread over the square [−1, 1]², red inside the circle x₁² + x₂² < 0.56 and blue outside. Train logistic regression twice, once on the features (x₁, x₂) and once on (x₁, x₂, x₁², x₂²), and compare the test accuracies.

<details><summary>Hint</summary>

Reuse the training loop from the example with a feature matrix built by `np.column_stack`. Train for a few thousand steps with η = 1.

</details>

In [ ]:
# Your code here

## 7.2 Encoding data

### Angle encoding and amplitude encoding

Angle encoding turns the point (x₁, x₂) into the angles θ = π(x₁ + 1)/2 and φ = πx₂ and prepares Ry(θ), then Rz(φ), on |0⟩. Amplitude encoding divides a list of four numbers by its length and uses them as the amplitudes of two qubits.

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def angle_encode(x1, x2):
    theta, phi = np.pi * (x1 + 1) / 2, np.pi * x2
    return rz(phi) @ ry(theta) @ np.array([1, 0])


for point in [(-1, 0), (0.5, -0.5), (1, 0.25)]:
    psi = angle_encode(*point)
    r = np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])
    print(f"x = {point}:  Bloch vector {r.round(3) + 0}   P(0) = {abs(psi[0]) ** 2:.3f}")

x = np.array([0.8, -0.3, 0.5, 0.1])
print("amplitude encoding of", x, "->", (x / np.linalg.norm(x)).round(3), " original length (discarded):", np.linalg.norm(x).round(3))

The point (0.5, −0.5) lands at (0, −0.707, −0.707) with P(0) ≈ 0.15, as in the worked example. Amplitude encoding keeps the direction of the list but loses its length, 0.995.

### Exercise 7.2.1: Re-uploading adds frequencies

Build the one-qubit model f(x) = ⟨Z⟩ for W<sub>L</sub> Rx(x) ⋯ W<sub>1</sub> Rx(x) W<sub>0</sub>|0⟩, where the W are fixed random one-qubit gates and x is encoded L times. Sample f at 64 equally spaced values of x from 0 up to, but not including, 2π (`endpoint=False` in `np.linspace`), take the Fourier transform with `np.fft.rfft`, and print the size of the first six coefficients for L = 1, 2 and 3.

<details><summary>Hint</summary>

A random one-qubit gate can be Rz(a) Ry(b) Rz(c) with random angles. Divide the FFT by the number of samples to get the coefficients.

</details>

In [ ]:
# Your code here

### Exercise 7.2.2: When the angles wrap around

Encode x₁ = −1 and x₁ = +1 (with x₂ = 0) using the scaled angle θ = s·π(x₁ + 1)/2, for s = 1 and s = 2. How far apart are the two encoded states, measured by their overlap |⟨ψ|χ⟩|²?

<details><summary>Hint</summary>

Only Ry matters here, because x₂ = 0 makes φ = 0.

</details>

In [ ]:
# Your code here

## 7.3 Circuits as models

### A one-qubit model and its decision boundary

The model from the worked example: encode x with Ry(x), apply the trainable Ry(θ), and measure Z. The code simulates it, compares it with the formula f = cos(x + θ), and turns f into predictions with p(red) = (1 − f)/2.

In [ ]:
import numpy as np


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def model(x, theta):
    psi = ry(theta) @ ry(x) @ np.array([1, 0])
    return psi[0] ** 2 - psi[1] ** 2                 # <Z>


theta = np.pi / 2 - 0.3
for x in [-1.0, 0.0, 0.25, 0.35, 1.0]:
    f = model(x, theta)
    p_red = (1 - f) / 2
    label = "blue" if f > 0 else "red"
    print(f"x = {x:+.2f}:  f = {f:+.3f}  cos(x + theta) = {np.cos(x + theta):+.3f}  p(red) = {p_red:.3f}  -> {label}")

With θ = π/2 − 0.3 the boundary f = 0 sits at x = 0.3: inputs below it are predicted blue and inputs above it red.

### Exercise 7.3.1: Rotating, then measuring Z, is measuring along a tilted axis

For a random one-qubit gate U and a random state |ψ⟩, check that ⟨ψ|U†ZU|ψ⟩ = n·r, where r is the Bloch vector of |ψ⟩ and n is the Bloch vector of U†|0⟩. This is the fact behind f(x) = n·r(x).

<details><summary>Hint</summary>

Build a random unitary as Rz(a) Ry(b) Rz(c), and reuse the `bloch` function from chapter 1.3.

</details>

In [ ]:
# Your code here

### Exercise 7.3.2: Why XOR is out of reach

Encode the four XOR corners (±0.5, ±0.5) with the angle encoding of chapter 7.2: red where x₁ and x₂ have the same sign, blue otherwise. Try 10,000 random measurement axes n with the prediction "blue if n·r > 0". What is the best accuracy any axis reaches?

<details><summary>Hint</summary>

The two red corners land at opposite points of the sphere, and so do the two blue ones.

</details>

In [ ]:
# Your code here

## 7.4 Gradients: the parameter-shift rule

### The parameter-shift rule, checked

The circuit from the chapter's panel: |0⟩ → Ry(0.9) → Rz(0.7) → Rx(θ), then measure Z. The parameter-shift rule [f(θ + π/2) − f(θ − π/2)]/2 is compared with the slope measured by a tiny finite difference on the exact simulation.

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def rot(P, t):
    return np.cos(t / 2) * I - 1j * np.sin(t / 2) * P


def f(theta):
    psi = rot(X, theta) @ rot(Z, 0.7) @ rot(Y, 0.9) @ np.array([1, 0])
    return np.vdot(psi, Z @ psi).real


for theta in [-1.4, 0.0, 0.8, 2.0]:
    shift = (f(theta + np.pi / 2) - f(theta - np.pi / 2)) / 2
    exact = (f(theta + 1e-6) - f(theta - 1e-6)) / 2e-6
    print(f"theta = {theta:+.1f}:  parameter shift {shift:+.6f}   true slope {exact:+.6f}")

The two columns agree to every printed digit: the shift π/2 is large, and the rule is still exact.

### Exercise 7.4.1: Parameter shift versus finite differences, with shot noise

On hardware every f is estimated from N shots. At θ = −1.4, estimate the gradient 300 times over, with N = 1,000 shots per evaluation, once with the parameter-shift rule and once with a finite difference of step h = 0.01. Compare the spread (standard deviation) of the two estimators.

<details><summary>Hint</summary>

With P(0) = (1 + f)/2, the number of 0 outcomes in N shots is `rng.binomial(N, (1 + f) / 2)`, and the estimate of f is 2 × count/N − 1.

</details>

In [ ]:
# Your code here

### Exercise 7.4.2: The chain rule for data-dependent angles

In a classifier the angle is often w·x + b. For the model f = ⟨Z⟩ after Ry(w·x + b)|0⟩, with x = 0.8, w = 1.3 and b = 0.3, compute ∂f/∂w and ∂f/∂b with the parameter-shift rule and the chain rule, and check them against finite differences.

<details><summary>Hint</summary>

Shift the whole angle by ±π/2 to get ∂f/∂(angle). Then ∂f/∂b is that number, and ∂f/∂w is x times it.

</details>

In [ ]:
# Your code here

## 7.5 Train a quantum classifier

### The training loop of chapter 7.5, in code

The seven points and the model f(x; θ) = cos(x + θ) from the "One step, every number" panel. Each step does a forward pass, computes the cross-entropy, gets ∂f/∂θ with the parameter-shift rule, multiplies by ∂L/∂f, and updates θ with η = 0.6.

In [ ]:
import numpy as np

xs = np.array([-1.4, -0.9, -0.4, 0.1, 0.6, 1.0, 1.4])
ys = np.array([0, 0, 0, 0, 1, 1, 1])           # 0 = blue, 1 = red


def f(x, theta):
    return np.cos(x + theta)                    # <Z> after Ry(x), then Ry(theta)


def loss_and_grad(theta):
    fx = f(xs, theta)
    p = (1 - fx) / 2                            # p(red)
    loss = -np.mean(ys * np.log(p) + (1 - ys) * np.log(1 - p))
    df = (f(xs, theta + np.pi / 2) - f(xs, theta - np.pi / 2)) / 2     # parameter shift
    dL_df = (ys / p - (1 - ys) / (1 - p)) / 2                           # from the loss formula
    return loss, np.mean(dL_df * df)


theta, eta = 0.0, 0.6
for step in range(41):
    loss, grad = loss_and_grad(theta)
    if step in (0, 1, 2, 5, 10, 20, 40):
        accuracy = np.mean((f(xs, theta) < 0) == ys)
        print(f"step {step:2d}: theta = {theta:.3f}  loss = {loss:.4f}  gradient = {round(grad, 4) + 0:+.4f}  accuracy {accuracy:.0%}")
    theta -= eta * grad
print(f"boundary at x = pi/2 - theta = {np.pi / 2 - theta:.3f}")

The first step takes θ from 0 to 0.662, as in the panel, and θ settles near 1.279, which puts the boundary at x = π/2 − θ ≈ 0.29, between the last blue point and the first red one.

### Exercise 7.5.1: Train a classifier with a trainable weight

Points x between −1 and 1 are blue when |x| < 0.5 and red otherwise. Train the model f = ⟨Z⟩ after Ry(w·x + b)|0⟩, which is cos(wx + b), starting from w = 1 and b = 0. Use the parameter-shift rule with the chain rule for both w and b, the cross-entropy loss, and plain gradient descent. Print the loss and accuracy as training goes, and the final w and b.

<details><summary>Hint</summary>

∂f/∂(angle) comes from the shift rule; then ∂f/∂b = ∂f/∂(angle) and ∂f/∂w = x · ∂f/∂(angle). Average ∂L/∂f × ∂f/∂w over the points to get the gradient for w.

</details>

In [ ]:
# Your code here

### Exercise 7.5.2: What a gradient costs on hardware

Write `evaluations(gate_angles, examples, epochs)`, the number of circuit runs for training with the parameter-shift rule: one forward run plus two shifted runs per gate angle, for every example in every epoch. Reproduce the chapter's count for one qubit with three layers (9 gate angles), 80 examples and 300 epochs, and turn it into shots at 1,000 shots per run.

In [ ]:
# Your code here

## 7.6 Quantum kernels

### A quantum kernel and a kernel model

The kernel of the chapter's one-dimensional panel: x is encoded as Ry(cπx)|0⟩ on each of two qubits, and k(x, x′) = |⟨φ(x)|φ(x′)⟩|². The code checks the closed form cos²(cπ(x − x′)/2) per qubit, then fits a kernel model f(x) = Σ αᵢ k(x, xᵢ) by kernel ridge regression, solving (K + λI)α = y.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=18)
c, copies, lam = 0.6, 2, 0.01


def encode(x):
    one = np.array([np.cos(c * np.pi * x / 2), np.sin(c * np.pi * x / 2)])   # Ry(c pi x)|0>
    psi = np.array([1.0])
    for _ in range(copies):
        psi = np.kron(psi, one)
    return psi


def kernel(a, b):
    return abs(np.vdot(encode(a), encode(b))) ** 2


print("k(0.2, -0.5) from the states:", round(kernel(0.2, -0.5), 6),
      "  closed form:", round(np.cos(c * np.pi * 0.7 / 2) ** (2 * copies), 6))

x_train = np.linspace(-0.95, 0.95, 14) + rng.uniform(-0.04, 0.04, 14)
y_train = np.where(np.abs(x_train) < 0.45, 1.0, -1.0)            # +1 blue, -1 red
K = np.array([[kernel(a, b) for b in x_train] for a in x_train])
alpha = np.linalg.solve(K + lam * np.eye(len(K)), y_train)


def model(x):
    return sum(a * kernel(x, xi) for a, xi in zip(alpha, x_train))


train_ok = np.mean(np.sign([model(x) for x in x_train]) == y_train)
grid = np.linspace(-1, 1, 201)
grid_ok = np.mean(np.sign([model(x) for x in grid]) == np.where(np.abs(grid) < 0.45, 1, -1))
print(f"training points correct: {train_ok:.0%}    accuracy on the whole interval: {grid_ok:.0%}")

Every training point is classified correctly, and the model is right on almost all of the interval; the only errors sit right next to the class boundaries at |x| = 0.45. Only the weights α were trained; the circuit just measures similarity.

### Exercise 7.6.1: Kernel width: underfitting and memorizing

Take 30 training points between −1 and 1, blue for |x| < 0.45 and red otherwise, and flip each label with probability 0.15 to imitate noisy data. Train the kernel model with c = 0.6 and λ = 0.1 while encoding x on 1, 4, 16, 64 and 256 qubits (copies), which makes the kernel narrower and narrower. Print the accuracy on the noisy training labels and on the true labels of 401 fresh points.

<details><summary>Hint</summary>

With m copies the kernel is cos<sup>2m</sup>(cπ(x − x′)/2), so you never need to build the 2<sup>m</sup>-dimensional states.

</details>

In [ ]:
# Your code here

### Exercise 7.6.2: Kernel values shrink exponentially with qubits

Encode n random features, one per qubit, with Ry(πx) on each qubit, so k(x, x′) = Π cos²(π(x<sub>i</sub> − x′<sub>i</sub>)/2). For n = 1, 2, 4, 8, 16 and 32, print the average kernel value between 200 random pairs of inputs drawn from [−1, 1]<sup>n</sup>, and compare it with the exact average, 1/2<sup>n</sup>: each qubit's factor averages exactly 1/2.

<details><summary>Hint</summary>

The product formula lets you compute k without building any 2<sup>n</sup>-dimensional state.

</details>

In [ ]:
# Your code here

## 7.7 Barren plateaus

### Watch a barren plateau appear

The same experiment as the chapter's panel, in NumPy, with 10 layers. Random circuits (in every layer, Ry and Rz on every qubit, then a ladder of CZ gates) are drawn 200 times for each number of qubits. For each one, the parameter-shift rule gives the gradient of the global cost, P(all qubits read 0), with respect to the first angle, and the code prints the variance of those gradients.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=21)


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def apply(psi, U, q):
    return np.moveaxis(np.tensordot(U, psi, axes=([1], [q])), 0, q)


def cz_ladder(n):
    bits = np.indices([2] * n)                          # bits[q] = value of qubit q
    pairs = sum(bits[q] * bits[q + 1] for q in range(n - 1))
    return (-1.0) ** pairs                              # sign of every amplitude


def global_cost(angles, n, layers, ladder):
    psi = np.zeros([2] * n, dtype=complex)
    psi[(0,) * n] = 1
    k = 0
    for _ in range(layers):
        for q in range(n):
            psi = apply(psi, ry(angles[k]), q)
            psi = apply(psi, rz(angles[k + 1]), q)
            k += 2
        psi = psi * ladder
    return abs(psi[(0,) * n]) ** 2                      # P(00...0)


layers, samples = 10, 200
for n in range(2, 9):
    ladder = cz_ladder(n)
    grads = []
    for _ in range(samples):
        angles = rng.uniform(0, 2 * np.pi, 2 * n * layers)
        plus, minus = angles.copy(), angles.copy()
        plus[0] += np.pi / 2
        minus[0] -= np.pi / 2
        grads.append((global_cost(plus, n, layers, ladder) - global_cost(minus, n, layers, ladder)) / 2)
    print(f"n = {n}: variance of the gradient {np.var(grads):.2e}")

Each added qubit divides the variance by roughly the same factor, which is exponential decay: the landscape flattens fast. The simulator stores the state as an array with one axis of length 2 per qubit, so a one-qubit gate is a single `np.tensordot`; chapter 7.8 explains this trick.

### Exercise 7.7.1: Global versus local cost in shallow circuits

Repeat the experiment with only 2 layers, and measure two costs: the global cost P(all qubits read 0) and the local cost, the average over the qubits of P(that qubit reads 0). For n = 2 to 8, print the variance of the gradient of each.

<details><summary>Hint</summary>

From the probabilities p = |ψ|², P(qubit q reads 0) is `p.take(0, axis=q).sum()`.

</details>

In [ ]:
# Your code here

### Exercise 7.7.2: How many shots would a gradient need?

The example measured these gradient variances for 10-layer circuits of 2 to 8 qubits. Fit a straight line to log(variance) against n, extrapolate to 20, 30 and 50 qubits, and estimate the shots needed per evaluation as roughly 1/variance, since a gradient of typical size g = √variance needs about 1/g² shots to stand out from the noise.

<details><summary>Hint</summary>

`np.polyfit(n, np.log(variance), 1)` fits the line. Its slope is minus the logarithm of the factor by which the variance shrinks per qubit, so that factor is e<sup>−slope</sup>.

</details>

In [ ]:
# Your code here

## 7.8 Where to go next

Each exercise adds a feature to the NumPy simulator above. The solutions repeat the functions they need, so every one runs on its own.

### Exercise 7.8.1: Any controlled gate

Write `apply_controlled(psi, U, c, t, n)`, which applies the one-qubit gate U to qubit t only in the part of the state where qubit c is 1. Check that it agrees with `apply_cx` when U = X, then use it to apply a controlled-H to |+⟩|0⟩.

<details><summary>Hint</summary>

Copy the idea of `apply_cx`: select the slice where the control is 1 with `idx[c] = 1`, and apply U along the target's axis inside that slice with `np.tensordot`. Remember that the target's axis number drops by one when t > c.

</details>

In [ ]:
# Your code here

### Exercise 7.8.2: Measurement with collapse

Write `measure(psi, q, n, rng)`, which measures qubit q: it picks 0 or 1 with the Born-rule probability, sets every amplitude that disagrees to zero, renormalizes, and returns the outcome and the new state. Make a Bell pair, measure q0 and then q1, and repeat 1,000 times. How often do the two results agree?

<details><summary>Hint</summary>

Reshape the state to n axes; `psi.take(1, axis=q)` is the part where qubit q is 1, so P(1) is the sum of its squared magnitudes.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 7.1.1: Gradient descent by hand, then by computer

In [ ]:
theta, eta = 0.0, 0.1
for step in range(1, 9):
    grad = 2 * (theta - 3)
    theta = theta - eta * grad
    print(f"step {step}: theta = {theta:.4f}   distance to 3 = {3 - theta:.4f}")

Expected output:

```
step 1: theta = 0.6000   distance to 3 = 2.4000
step 2: theta = 1.0800   distance to 3 = 1.9200
step 3: theta = 1.4640   distance to 3 = 1.5360
step 4: theta = 1.7712   distance to 3 = 1.2288
step 5: theta = 2.0170   distance to 3 = 0.9830
step 6: theta = 2.2136   distance to 3 = 0.7864
step 7: theta = 2.3709   distance to 3 = 0.6291
step 8: theta = 2.4967   distance to 3 = 0.5033
```

Each step multiplies the distance to the minimum by 1 − 2η = 0.8, so it removes 20% of it. With η = 1 the factor would be −1 and θ would jump back and forth forever; with η above 1 it would run away.

### Solution 7.1.2: Features decide what a model can learn

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=12)


def make_circle(n):
    X = rng.uniform(-1, 1, size=(n, 2))
    return X, (np.sum(X ** 2, axis=1) < 0.56).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def train_and_test(features, steps=3000, lr=1.0):
    F_train, F_test = features(X_train), features(X_test)
    w, b = np.zeros(F_train.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(F_train @ w + b)
        w -= lr * F_train.T @ (p - y_train) / len(y_train)
        b -= lr * np.mean(p - y_train)
    return np.mean((sigmoid(F_test @ w + b) > 0.5) == y_test)


X_train, y_train = make_circle(200)
X_test, y_test = make_circle(200)
print(f"share of red points in the test set: {y_test.mean():.0%}")
print(f"features x1, x2:             test accuracy {train_and_test(lambda X: X):.0%}")
print(f"features x1, x2, x1^2, x2^2: test accuracy {train_and_test(lambda X: np.column_stack([X, X ** 2])):.0%}")

Expected output:

```
share of red points in the test set: 42%
features x1, x2:             test accuracy 58%
features x1, x2, x1^2, x2^2: test accuracy 98%
```

With x₁ and x₂ alone the boundary is a straight line, and the model can do little better than guessing the most common class. With the squared features the boundary can be a circle. In a quantum model, the encoding plays the role of these features.

### Solution 7.2.1: Re-uploading adds frequencies

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=13)
X = np.array([[0, 1], [1, 0]])


def rx(t):
    return np.cos(t / 2) * np.eye(2) - 1j * np.sin(t / 2) * X


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def random_gate():
    a, b, c = rng.uniform(0, 2 * np.pi, 3)
    return rz(a) @ ry(b) @ rz(c)


xs = np.linspace(0, 2 * np.pi, 64, endpoint=False)
for L in [1, 2, 3]:
    W = [random_gate() for _ in range(L + 1)]
    f = []
    for x in xs:
        psi = W[0] @ np.array([1, 0])
        for k in range(1, L + 1):
            psi = W[k] @ rx(x) @ psi
        f.append(abs(psi[0]) ** 2 - abs(psi[1]) ** 2)          # <Z>
    coeffs = np.abs(np.fft.rfft(f)) / len(xs)
    print(f"L = {L}: |c_k| for k = 0..5:", coeffs[:6].round(3))

Expected output:

```
L = 1: |c_k| for k = 0..5: [0.229 0.383 0.    0.    0.    0.   ]
L = 2: |c_k| for k = 0..5: [0.46  0.082 0.204 0.    0.    0.   ]
L = 3: |c_k| for k = 0..5: [0.185 0.347 0.092 0.144 0.    0.   ]
```

With L encoding gates, every coefficient above frequency L is zero (up to rounding). Each extra upload adds one more frequency, exactly as the chapter says.

### Solution 7.2.2: When the angles wrap around

In [ ]:
import numpy as np


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


for s in [1, 2]:
    a = ry(s * np.pi * (-1 + 1) / 2) @ [1, 0]
    b = ry(s * np.pi * (1 + 1) / 2) @ [1, 0]
    print(f"scale {s}: overlap of the encodings of x1 = -1 and x1 = +1: {abs(np.vdot(a, b)) ** 2:.3f}")

Expected output:

```
scale 1: overlap of the encodings of x1 = -1 and x1 = +1: 0.000
scale 2: overlap of the encodings of x1 = -1 and x1 = +1: 1.000
```

With s = 1 the two ends of the input range go to opposite poles, overlap 0. With s = 2 the angle runs a full turn and both ends land on the same point, overlap 1: no model could tell them apart. Choosing the scale is choosing how far apart inputs look.

### Solution 7.3.1: Rotating, then measuring Z, is measuring along a tilted axis

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
rng = np.random.default_rng(seed=14)


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def bloch(psi):
    return np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])


for _ in range(3):
    a, b, c = rng.uniform(0, 2 * np.pi, 3)
    U = rz(a) @ ry(b) @ rz(c)
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    psi = v / np.linalg.norm(v)
    f = np.vdot(psi, U.conj().T @ Z @ U @ psi).real
    n = bloch(U.conj().T @ np.array([1, 0]))
    print(f"<psi|U^dagger Z U|psi> = {f:+.4f}    n . r = {n @ bloch(psi):+.4f}")

Expected output:

```
<psi|U^dagger Z U|psi> = -0.6284    n . r = -0.6284
<psi|U^dagger Z U|psi> = -0.0499    n . r = -0.0499
<psi|U^dagger Z U|psi> = +0.8716    n . r = +0.8716
```

### Solution 7.3.2: Why XOR is out of reach

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=15)


def encode(x1, x2):
    theta, phi = np.pi * (x1 + 1) / 2, np.pi * x2
    return np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])


corners = [(-0.5, -0.5), (0.5, 0.5), (-0.5, 0.5), (0.5, -0.5)]
labels = np.array([1, 1, 0, 0])                    # red (1) when the signs agree
R = np.array([encode(*c) for c in corners])
print("encoded corners:\n", R.round(3) + 0)

best = 0
for _ in range(10_000):
    n = rng.normal(size=3)
    predictions = (R @ n < 0).astype(int)          # f = n.r < 0 means red
    best = max(best, np.mean(predictions == labels))
print(f"best accuracy over 10,000 axes: {best:.0%}")

Expected output:

```
encoded corners:
 [[ 0.    -0.707  0.707]
 [ 0.     0.707 -0.707]
 [ 0.     0.707  0.707]
 [ 0.    -0.707 -0.707]]
best accuracy over 10,000 axes: 50%
```

Every axis gets exactly two of the four corners right. Because n·(−r) = −n·r, a plane through the centre always puts opposite points on opposite sides, so it can never put both red corners on the red side.

### Solution 7.4.1: Parameter shift versus finite differences, with shot noise

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=16)
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def rot(P, t):
    return np.cos(t / 2) * I - 1j * np.sin(t / 2) * P


def f(theta):
    psi = rot(X, theta) @ rot(Z, 0.7) @ rot(Y, 0.9) @ np.array([1, 0])
    return np.vdot(psi, Z @ psi).real


def f_shots(theta, N):
    zeros = rng.binomial(N, (1 + f(theta)) / 2)
    return 2 * zeros / N - 1


theta, N, h = -1.4, 1000, 0.01
true = (f(theta + np.pi / 2) - f(theta - np.pi / 2)) / 2
shift = [(f_shots(theta + np.pi / 2, N) - f_shots(theta - np.pi / 2, N)) / 2 for _ in range(300)]
finite = [(f_shots(theta + h, N) - f_shots(theta - h, N)) / (2 * h) for _ in range(300)]
print(f"true gradient {true:+.3f}")
print(f"parameter shift:   mean {np.mean(shift):+.3f}   spread {np.std(shift):.3f}")
print(f"finite difference: mean {np.mean(finite):+.3f}   spread {np.std(finite):.3f}")

Expected output:

```
true gradient +0.698
parameter shift:   mean +0.698   spread 0.016
finite difference: mean +0.578   spread 1.980
```

The finite difference divides the shot noise by 2h = 0.02 instead of by 2, so its spread is more than a hundred times larger, far bigger than the gradient itself.

### Solution 7.4.2: The chain rule for data-dependent angles

In [ ]:
import numpy as np


def f_of_angle(a):
    return np.cos(a)                          # <Z> after Ry(a)|0>


x, w, b = 0.8, 1.3, 0.3
angle = w * x + b
d_angle = (f_of_angle(angle + np.pi / 2) - f_of_angle(angle - np.pi / 2)) / 2
df_dw, df_db = x * d_angle, d_angle

eps = 1e-6
num_dw = (f_of_angle((w + eps) * x + b) - f_of_angle((w - eps) * x + b)) / (2 * eps)
num_db = (f_of_angle(w * x + b + eps) - f_of_angle(w * x + b - eps)) / (2 * eps)
print(f"df/dw: chain rule {df_dw:+.6f}   finite difference {num_dw:+.6f}")
print(f"df/db: chain rule {df_db:+.6f}   finite difference {num_db:+.6f}")

Expected output:

```
df/dw: chain rule -0.778788   finite difference -0.778788
df/db: chain rule -0.973485   finite difference -0.973485
```

### Solution 7.5.1: Train a classifier with a trainable weight

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=17)
xs = rng.uniform(-1, 1, 60)
ys = (np.abs(xs) >= 0.5).astype(float)          # 1 = red, 0 = blue


def f(angle):
    return np.cos(angle)                         # <Z> after Ry(angle)|0>


w, b, eta = 1.0, 0.0, 0.5
for step in range(301):
    angle = w * xs + b
    fx = f(angle)
    p = np.clip((1 - fx) / 2, 1e-9, 1 - 1e-9)    # p(red)
    loss = -np.mean(ys * np.log(p) + (1 - ys) * np.log(1 - p))
    d_angle = (f(angle + np.pi / 2) - f(angle - np.pi / 2)) / 2     # parameter shift
    dL_df = (ys / p - (1 - ys) / (1 - p)) / 2
    grad_w = np.mean(dL_df * d_angle * xs)
    grad_b = np.mean(dL_df * d_angle)
    if step % 60 == 0:
        print(f"step {step:3d}: loss {loss:.3f}  accuracy {np.mean((fx < 0) == ys):.0%}  w = {w:.3f}  b = {b:+.3f}")
    w, b = w - eta * grad_w, b - eta * grad_b
print(f"final model: f = cos({w:.3f} x {'+' if b >= 0 else '-'} {abs(b):.3f}),  compare with pi = {np.pi:.3f}")

Expected output:

```
step   0: loss 0.894  accuracy 57%  w = 1.000  b = +0.000
step  60: loss 0.181  accuracy 97%  w = 3.420  b = -0.105
step 120: loss 0.181  accuracy 97%  w = 3.430  b = -0.103
step 180: loss 0.181  accuracy 97%  w = 3.430  b = -0.103
step 240: loss 0.181  accuracy 97%  w = 3.430  b = -0.103
step 300: loss 0.181  accuracy 97%  w = 3.430  b = -0.103
final model: f = cos(3.430 x - 0.103),  compare with pi = 3.142
```

Training moves w from 1 to about 3.4 and b to about −0.1, close to the ideal w = π, b = 0, for which cos(πx) changes sign exactly at |x| = 0.5. The trainable weight rescales the encoding, the same trick as the trainable data scaling in the chapter's classifier. The few remaining errors are points very close to the boundary.

### Solution 7.5.2: What a gradient costs on hardware

In [ ]:
def evaluations(gate_angles, examples, epochs):
    return (1 + 2 * gate_angles) * examples * epochs


runs = evaluations(gate_angles=9, examples=80, epochs=300)
print(f"circuit evaluations: {runs:,}")
print(f"shots at 1,000 per evaluation: {runs * 1000:,}")
for qubits, layers in [(2, 3), (4, 6), (10, 10)]:
    angles = 3 * qubits * layers                 # three rotations per qubit per layer
    print(f"{qubits} qubits, {layers} layers: {evaluations(angles, 80, 300):,} evaluations")

Expected output:

```
circuit evaluations: 456,000
shots at 1,000 per evaluation: 456,000,000
2 qubits, 3 layers: 888,000 evaluations
4 qubits, 6 layers: 3,480,000 evaluations
10 qubits, 10 layers: 14,424,000 evaluations
```

### Solution 7.6.1: Kernel width: underfitting and memorizing

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=22)
x_train = rng.uniform(-1, 1, 30)
y_clean = np.where(np.abs(x_train) < 0.45, 1.0, -1.0)
flipped = rng.random(30) < 0.15
y_train = np.where(flipped, -y_clean, y_clean)               # noisy labels
grid = np.linspace(-1, 1, 401)
y_grid = np.where(np.abs(grid) < 0.45, 1.0, -1.0)
print("labels flipped:", flipped.sum(), "of 30")

c, lam = 0.6, 0.1
for copies in [1, 4, 16, 64, 256]:
    def k(a, b):
        return np.cos(c * np.pi * (a - b) / 2) ** (2 * copies)
    K = k(x_train[:, None], x_train[None, :])
    alpha = np.linalg.solve(K + lam * np.eye(30), y_train)

    def f(x):
        return k(np.asarray(x)[:, None], x_train[None, :]) @ alpha
    train = np.mean(np.sign(f(x_train)) == y_train)
    test = np.mean(np.sign(f(grid)) == y_grid)
    print(f"{copies:3d} cop{'y  ' if copies == 1 else 'ies'}: training accuracy {train:.0%}   accuracy on fresh points {test:.1%}")

Expected output:

```
labels flipped: 3 of 30
  1 copy  : training accuracy 80%   accuracy on fresh points 84.8%
  4 copies: training accuracy 90%   accuracy on fresh points 93.0%
 16 copies: training accuracy 90%   accuracy on fresh points 99.5%
 64 copies: training accuracy 90%   accuracy on fresh points 97.0%
256 copies: training accuracy 97%   accuracy on fresh points 89.8%
```

One copy gives a wide, smooth kernel that underfits. Around 16 copies the model ignores the flipped labels and gets 399 of the 401 fresh points right; the two it misses sit exactly on the class boundary, x = ±0.45. With 256 copies the kernel is so narrow that the model memorizes the training points, including the wrong labels, and does worse on new data: the training accuracy goes up while the test accuracy goes down.

### Solution 7.6.2: Kernel values shrink exponentially with qubits

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=20)
for n in [1, 2, 4, 8, 16, 32]:
    x = rng.uniform(-1, 1, size=(200, n))
    y = rng.uniform(-1, 1, size=(200, n))
    k = np.prod(np.cos(np.pi * (x - y) / 2) ** 2, axis=1)
    print(f"n = {n:2d} qubits: average of 200 pairs {k.mean():.2e}   exact average {0.5 ** n:.2e}   median {np.median(k):.2e}")

Expected output:

```
n =  1 qubits: average of 200 pairs 5.05e-01   exact average 5.00e-01   median 4.79e-01
n =  2 qubits: average of 200 pairs 2.81e-01   exact average 2.50e-01   median 1.90e-01
n =  4 qubits: average of 200 pairs 6.34e-02   exact average 6.25e-02   median 9.11e-03
n =  8 qubits: average of 200 pairs 3.93e-03   exact average 3.91e-03   median 2.06e-05
n = 16 qubits: average of 200 pairs 3.07e-06   exact average 1.53e-05   median 3.32e-10
n = 32 qubits: average of 200 pairs 1.91e-12   exact average 2.33e-10   median 9.46e-20
```

The exact average halves with every added qubit, which is exponential decay. For many qubits the average of 200 pairs comes out too low: the true average is carried by rare pairs that happen to be close, while most pairs have far smaller kernel values, as the median shows. With 32 qubits two different inputs look almost completely unlike each other, and estimating such tiny kernel values from shots would take an enormous number of measurements. This is exponential concentration.

### Solution 7.7.1: Global versus local cost in shallow circuits

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=23)


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def apply(psi, U, q):
    return np.moveaxis(np.tensordot(U, psi, axes=([1], [q])), 0, q)


def costs(angles, n, layers, ladder):
    psi = np.zeros([2] * n, dtype=complex)
    psi[(0,) * n] = 1
    k = 0
    for _ in range(layers):
        for q in range(n):
            psi = apply(psi, ry(angles[k]), q)
            psi = apply(psi, rz(angles[k + 1]), q)
            k += 2
        psi = psi * ladder
    p = np.abs(psi) ** 2
    global_cost = p[(0,) * n]
    local_cost = np.mean([p.take(0, axis=q).sum() for q in range(n)])
    return np.array([global_cost, local_cost])


layers, samples = 2, 200
for n in range(2, 9):
    bits = np.indices([2] * n)
    ladder = (-1.0) ** sum(bits[q] * bits[q + 1] for q in range(n - 1))
    grads = []
    for _ in range(samples):
        angles = rng.uniform(0, 2 * np.pi, 2 * n * layers)
        plus, minus = angles.copy(), angles.copy()
        plus[0] += np.pi / 2
        minus[0] -= np.pi / 2
        grads.append((costs(plus, n, layers, ladder) - costs(minus, n, layers, ladder)) / 2)
    var_global, var_local = np.var(grads, axis=0)
    print(f"n = {n}: variance of the gradient, global cost {var_global:.2e}   local cost {var_local:.2e}")

Expected output:

```
n = 2: variance of the gradient, global cost 3.05e-02   local cost 2.23e-02
n = 3: variance of the gradient, global cost 1.02e-02   local cost 9.40e-03
n = 4: variance of the gradient, global cost 4.22e-03   local cost 5.30e-03
n = 5: variance of the gradient, global cost 1.58e-03   local cost 3.42e-03
n = 6: variance of the gradient, global cost 5.09e-04   local cost 2.32e-03
n = 7: variance of the gradient, global cost 6.13e-05   local cost 1.78e-03
n = 8: variance of the gradient, global cost 1.26e-05   local cost 1.21e-03
```

In this shallow circuit the global cost's variance still shrinks by a large factor with every qubit, while the local cost's variance shrinks only slowly. Choosing local costs is one of the remedies in the chapter's table.

### Solution 7.7.2: How many shots would a gradient need?

In [ ]:
import numpy as np

n = np.arange(2, 9)
variance = np.array([2.48e-02, 6.71e-03, 1.93e-03, 5.03e-04, 1.36e-04, 5.21e-05, 7.64e-06])   # from the example
slope, intercept = np.polyfit(n, np.log(variance), 1)
print(f"variance shrinks by a factor of about {np.exp(-slope):.1f} per added qubit")
for qubits in [20, 30, 50]:
    v = np.exp(intercept + slope * qubits)
    print(f"{qubits} qubits: variance about {v:.1e}, so about {1 / v:.1e} shots per gradient evaluation")

Expected output:

```
variance shrinks by a factor of about 3.7 per added qubit
20 qubits: variance about 1.5e-12, so about 6.5e+11 shots per gradient evaluation
30 qubits: variance about 3.2e-18, so about 3.1e+17 shots per gradient evaluation
50 qubits: variance about 1.4e-29, so about 7.2e+28 shots per gradient evaluation
```

The fitted factor is close to 4 per qubit. At 50 qubits a single gradient component would need more shots than any machine could ever run: that is the barren plateau as a practical wall.

### Solution 7.8.1: Any controlled gate

In [ ]:
import numpy as np


def zero_state(n):
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1.0
    return psi


def apply_1q(psi, U, q, n):
    psi = psi.reshape([2] * n)
    psi = np.tensordot(U, psi, axes=([1], [q]))
    psi = np.moveaxis(psi, 0, q)
    return psi.reshape(-1)


def apply_cx(psi, c, t, n):
    psi = psi.reshape([2] * n).copy()
    idx = [slice(None)] * n
    idx[c] = 1
    t_axis = t if t < c else t - 1
    psi[tuple(idx)] = np.flip(psi[tuple(idx)], axis=t_axis)
    return psi.reshape(-1)


def apply_controlled(psi, U, c, t, n):
    psi = psi.reshape([2] * n).copy()
    idx = [slice(None)] * n
    idx[c] = 1                                       # the part where the control is 1
    t_axis = t if t < c else t - 1                   # axis c is gone from that slice
    block = np.tensordot(U, psi[tuple(idx)], axes=([1], [t_axis]))
    psi[tuple(idx)] = np.moveaxis(block, 0, t_axis)
    return psi.reshape(-1)


X = np.array([[0, 1], [1, 0]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
rng = np.random.default_rng(seed=24)
v = rng.normal(size=8) + 1j * rng.normal(size=8)
v /= np.linalg.norm(v)
same = all(np.allclose(apply_controlled(v, X, c, t, 3), apply_cx(v, c, t, 3))
           for c in range(3) for t in range(3) if c != t)
print("agrees with apply_cx for every control and target:", same)

psi = apply_1q(zero_state(2), H, 0, 2)               # |+>|0>
print("controlled-H on |+>|0>:", apply_controlled(psi, H, 0, 1, 2).real.round(4))

Expected output:

```
agrees with apply_cx for every control and target: True
controlled-H on |+>|0>: [0.7071 0.     0.5    0.5   ]
```

Controlled-H on |+⟩|0⟩ leaves the |0⟩ branch alone and turns the |1⟩ branch into |1⟩|+⟩, so the amplitudes are 1/√2, 0, 1/2 and 1/2.

### Solution 7.8.2: Measurement with collapse

In [ ]:
import numpy as np


def zero_state(n):
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1.0
    return psi


def apply_1q(psi, U, q, n):
    psi = psi.reshape([2] * n)
    psi = np.tensordot(U, psi, axes=([1], [q]))
    psi = np.moveaxis(psi, 0, q)
    return psi.reshape(-1)


def apply_cx(psi, c, t, n):
    psi = psi.reshape([2] * n).copy()
    idx = [slice(None)] * n
    idx[c] = 1
    t_axis = t if t < c else t - 1
    psi[tuple(idx)] = np.flip(psi[tuple(idx)], axis=t_axis)
    return psi.reshape(-1)


def measure(psi, q, n, rng):
    psi = psi.reshape([2] * n).copy()
    p1 = np.sum(np.abs(psi.take(1, axis=q)) ** 2)
    outcome = int(rng.random() < p1)
    idx = [slice(None)] * n
    idx[q] = 1 - outcome
    psi[tuple(idx)] = 0                              # remove the branch that did not happen
    psi = psi.reshape(-1)
    return outcome, psi / np.linalg.norm(psi)


H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
rng = np.random.default_rng(seed=25)
agree, ones = 0, 0
for _ in range(1000):
    psi = apply_cx(apply_1q(zero_state(2), H, 0, 2), 0, 1, 2)     # Bell pair
    m0, psi = measure(psi, 0, 2, rng)
    m1, psi = measure(psi, 1, 2, rng)
    agree += m0 == m1
    ones += m0
print(f"results agreed in {agree} of 1000 runs; q0 read 1 in {ones} runs")

Expected output:

```
results agreed in 1000 of 1000 runs; q0 read 1 in 483 runs
```

The two bits agree every time, and each value appears about half the time: the perfect correlation of the Bell state, now produced by your own simulator.